# Session 2 — From API to Application
### System prompts, structured outputs, function calling

**AI Systems in Practice** · HEC Paris · Lab (1h45)

---

You are the AI lead at **Velora**, a European e-bike retailer. 

Support receives ~2 000 emails per week. A human reads each one and retypes the key facts into the
ticketing system. 

You have been asked whether an LLM can do the extraction, and **what it would take to trust it**.

By the end of this notebook you will have:

1. measured how often a well-written prompt still produces unusable output;
2. replaced the prompt with a **contract** (a schema) and measured the difference;
3. found the three places where the model is confidently wrong, and fixed two of them;
4. let the model **call your code** instead of just describing what it would do.

**Read `DATA_SPEC.md` before you start.** It is the business specification. Your
extractor is judged against it, not against your intuition.

| | | |
|---|---|---|
| 0. Setup | 10 min | keys, data, smoke test |
| A. The baseline that fails | 15 min | free-text JSON, measured |
| B. The contract | 25 min | schema + system prompt + first score |
| C. Where it is confidently wrong | 25 min | error analysis, v2, re-score |
| D. Letting the model act | 20 min | function calling |
| E. The email that attacks you | 10 min | indirect prompt injection |
| F. Bonus + write-up | 10 min | free |

**Deliverable:** the executed notebook, plus a one-page
report (due 23:59 tonight) — see the last cell: due next Thursday evening

## 0. Setup

You have been given **one** key, the same OpenRouter key as session 1.


| model | who built it | reached as |
|---|---|---|
| **DeepSeek** | open weights | `deepseek/deepseek-chat-v3.1` |

Session 1 fixed the prompt and varied the model. This session does the opposite: **one
model, and everything around it varies** — the system prompt, the schema, the tools.
Vary two things at once and you cannot attribute the difference to either.

That one model is reached the way every other one is: an OpenAI-compatible `base_url`
and a key.

In [26]:
!pip install -q openai pydantic

import json
import os
import time
import urllib.request
from getpass import getpass
from typing import Optional, Literal
from pydantic import BaseModel, Field
from openai import OpenAI
import pandas as pd
import threading
from concurrent.futures import ThreadPoolExecutor


print("ready")

ready


In [9]:
# Everything this lab needs is in one public folder, downloaded on the fly.
BASE = "https://raw.githubusercontent.com/vorpax/velora-labs/ref/head/main/s2/"

FILES   = ["emails.json", "ground_truth_dev.json", "orders_db.json"]

for name in FILES:
    urllib.request.urlretrieve(BASE + name, name)

print("downloaded:", sorted(p for p in os.listdir() if not p.startswith(".")))

HTTPError: HTTP Error 404: Not Found

In [11]:
EMAILS   = json.load(open("emails.json", encoding="utf-8"))            # 50 emails, no labels
GT_DEV   = json.load(open("ground_truth_dev.json", encoding="utf-8"))  # 35 labels: your working set
ORDERS   = json.load(open("orders_db.json", encoding="utf-8"))         # for Part D

BY_ID    = {e["id"]: e for e in EMAILS}
GT_BY_ID = {g["id"]: g for g in GT_DEV}
DEV_IDS  = sorted(GT_BY_ID)                    # the 35 you can score yourself

print(f"{len(EMAILS)} emails, {len(DEV_IDS)} labelled for development")
print(json.dumps(EMAILS[0], indent=2, ensure_ascii=False))

50 emails, 35 labelled for development
{
  "id": "E01",
  "received_at": "2026-03-09",
  "subject": "Refund request - helmet wrong size",
  "body": "Hello,\n\nI ordered a Velora Aero helmet in size M, order VLR-2026-04412, and it arrived on\n5 March. It is far too small - I measured my head at 59 cm which should be a L.\n\nI would like a refund of the 89.90 EUR rather than an exchange, as I have already\nbought a replacement elsewhere.\n\nBest regards,\nAnna Kowalski"
}


In [12]:
# ------------------------------------------------------- one key, one client
BASE_URL = "https://openrouter.ai/api/v1"

MODEL = "deepseek/deepseek-chat-v3.1"


client = OpenAI(api_key=getpass("OpenRouter API key: ").strip(),
                base_url=BASE_URL, max_retries=5)

print(f"model={MODEL}  endpoint={client.base_url}")

model=deepseek/deepseek-chat-v3.1  endpoint=https://openrouter.ai/api/v1/


In [13]:
# ------------------------------------------------------------------ test
t0 = time.time()
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer in exactly three words."},
        {"role": "user",   "content": "What is a transformer?"},
    ],
    temperature=0,
)
print(r.choices[0].message.content)
print(f"{time.time()-t0:.2f}s · {r.usage.prompt_tokens} in / {r.usage.completion_tokens} out")

A neural network architecture.
3.13s · 15 in / 6 out


In [15]:
print(r)
print(r.usage)
print(r.model)
print(r.created)


ChatCompletion(id='gen-1789029110-M37aqcEvBRXbE2mNBUha', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='A neural network architecture.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1789029110, model='deepseek/deepseek-chat-v3.1', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=6, prompt_tokens=15, total_tokens=21, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=9.9495e-06, is_byok=False, cost_details={'upstream_inference_cost': 1.005e-05, 'upstream_inference_prompt_cost': 4.05e-06, 'upstream_inference_completions_cost': 6e-06})

> **Stop and look at that response object.** `choices[0].message.content` is a string.
> `usage` is a token count. There is no session, no memory, no state. The API is
> stateless: identical inputs give the same behaviour, and nothing carries between
> calls. Anything that looks like memory in ChatGPT is your history being re-sent,
> and re-billed, on every turn.

### What JSON is:

JSON is a text format for representing structured data: objects, lists, numbers, strings in a way any language (for instance : Python, Go, Typescript) can read.

In other terms, it is what a Python dictionary looks like once it's been flattened into plain text so it can travel over a network.

## A. The baseline that fails · 15 min

The obvious approach: ask nicely for JSON. It is easy, seems the obvious right way but it doesn't work systematically.

Run the next cell. It uses a *good* prompt — explicit fields, an example, an
instruction not to add prose — on 12 emails.

In [16]:
def as_user_message(email):
    return json.dumps({k: email[k] for k in ("received_at", "subject", "body")},
                      ensure_ascii=False) # We turn the email into a JSON string.

print(BY_ID[DEV_IDS[0]]) # It has everything, including internal IDs
print(as_user_message(BY_ID[DEV_IDS[0]])) # Drop the ID to make it look like a proper user message for the model

{'id': 'E01', 'received_at': '2026-03-09', 'subject': 'Refund request - helmet wrong size', 'body': 'Hello,\n\nI ordered a Velora Aero helmet in size M, order VLR-2026-04412, and it arrived on\n5 March. It is far too small - I measured my head at 59 cm which should be a L.\n\nI would like a refund of the 89.90 EUR rather than an exchange, as I have already\nbought a replacement elsewhere.\n\nBest regards,\nAnna Kowalski'}
{"received_at": "2026-03-09", "subject": "Refund request - helmet wrong size", "body": "Hello,\n\nI ordered a Velora Aero helmet in size M, order VLR-2026-04412, and it arrived on\n5 March. It is far too small - I measured my head at 59 cm which should be a L.\n\nI would like a refund of the 89.90 EUR rather than an exchange, as I have already\nbought a replacement elsewhere.\n\nBest regards,\nAnna Kowalski"}


In [17]:

NAIVE_PROMPT = """You are a support triage assistant for Velora, an e-bike retailer.
Read the customer email and return the following information as JSON:
customer_name, order_id, amount, currency, incident_date (YYYY-MM-DD), category, priority.

Categories: refund_request, delivery_issue, billing_error, product_defect, cancellation, general_inquiry.
Priorities: low, medium, high, urgent.

Example: {"customer_name": "Jane Doe", "order_id": "VLR-2026-01234", "amount": 99.0,
"currency": "EUR", "incident_date": "2026-01-15", "category": "delivery_issue", "priority": "medium"}

Return only the JSON object. Do not add any explanation."""


rows = [] # A list of dicts, one per email, to be turned into a DataFrame.
for eid in DEV_IDS[:12]: # We only do the first 12 for speed.
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": NAIVE_PROMPT},
                  {"role": "user",   "content": as_user_message(BY_ID[eid])}],
    )
    raw = r.choices[0].message.content # Get the output string. It should be a JSON object, but we don't know for sure.
    try: # We try to parse it as JSON. If it fails, we record the error type.
        json.loads(raw)  # We don't care about the result, just whether it parses. Hence the parsing.
        err = None
    except Exception as e:
        err = type(e).__name__
    rows.append({"id": eid, "parsed": err is None, "error": err,
                 "first_40_chars": repr(raw[:40])}) # We record the first 40 characters of the output for debugging.

df_naive = pd.DataFrame(rows) # Turn the list of dicts into a DataFrame for easier analysis.
print(f"parse success: {df_naive.parsed.mean():.0%}  ({df_naive.parsed.sum()}/12)")
df_naive

parse success: 100%  (12/12)


,id,parsed,error,first_40_chars
0,E01,True,None,"'{\n ""customer_name"": ""Anna Kowalski"",\n '"
1,E03,True,None,"'{""customer_name"": ""Sofia Marchetti"", ""or'"
2,E04,True,None,"'{""customer_name"": ""Juhani Virtanen"", ""or'"
3,E05,True,None,"'{\n ""customer_name"": ""Thomas Berger"",\n '"
4,E06,True,None,"'{\n ""customer_name"": ""Elena Popescu"",\n '"
5,E07,True,None,"'{\n ""customer_name"": ""Pieter de Vries"",\n'"
6,E08,True,None,"'{\n ""customer_name"": ""Clara Jimenez"",\n '"
7,E10,True,None,"'{\n ""customer_name"": ""Bruno Almeida"",\n '"
8,E11,True,None,"'{""customer_name"": ""Isabelle Roy"", ""order'"
9,E13,True,None,"'{\n ""customer_name"": ""Antoine Girard"",\n '"


**Answer these before moving on — they go in your report.**

1. What was your parse success rate? Run the cell a second time. Is it the same?
2. For any failure, look at `first_40_chars`. What did the model actually emit?
3. Suppose the rate is 97%. Velora processes 2 000 emails per week. How many broken
   tickets per year? Who notices, and how?
4. A colleague proposes: "wrap it in a try/except and retry three times."
      - What class of error does that fix, and what class does it leave untouched? 
      - hint: being able to process/parse doesn't mean it is correct

Keep the answer to 3 in mind. **The interesting failures are not the ones that crash.**

## B. The contract · 25 min

Two things change now.

**The schema.** Instead of describing the output in prose, you declare it as a type.
The provider compiles (converts) it into a constraint on decoding, so tokens that would break the
schema are never sampled (picked).
Structural validity stops being probabilistic (at random).

**The system prompt.** It stops being a guide about format and becomes a guide on the context.

These two ensure some separation of concerns (important!) :

| the schema guarantees | the prompt is still responsible |
|---|---|
| every key is present | the right value is in it |
| types are correct | the right one of two candidates was picked |
| enums are in range | the right enum member was chosen |
| `null` is allowed where you said so | `null` is used when the fact is absent |

A schema buys you **shape, not truth**. 
Most of your remaining errors will be semantic, and you will not see them unless you measure.

In [ ]:
# =============================== TODO 1 ======================================
# Declare the extraction schema, following DATA_SPEC.md:
#
# The overall types are already right. Your work is the two enums and
# every `description="..."`: the descriptions are sent to the model on every call.
# Each descriptions should carry the spec's rule for that field (which of two amounts? which date? null when?).
#
# Why the types look the way they do:
#   - a field that may be absent is `Optional[...]` WITHOUT a default value
#     (required-but-nullable)
#   - enums are Literal[...], spelled exactly as in the spec
#   - dates are plain `str` in "YYYY-MM-DD" form, not date objects

Example_enum = Literal["Foo", "Bar", "Baz"]  # <- example of a three-value enum. (3 options : Foo, Bar, Baz)

Category = Literal["..."]   # <- the six categories, as spelled in the spec
Priority = Literal["..."]   # <- the four priorities, as spelled in the spec

class Ticket(BaseModel):
    customer_name: Optional[str]                        = Field(description="...")
    order_id:      Optional[str]                        = Field(description="...")
    amount:        Optional[float]                      = Field(description="...")
    currency:      Optional[Literal["EUR", "USD", "GBP"]] = Field(description="...")
    incident_date: Optional[str]                        = Field(description="...")
    category:      Category                             = Field(description="...")
    priority:      Priority                             = Field(description="...")

print(json.dumps(Ticket.model_json_schema(), indent=2)[:600])

{
  "properties": {
    "customer_name": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "description": "...",
      "title": "Customer Name"
    },
    "order_id": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "description": "...",
      "title": "Order Id"
    },
    "amount": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "description": "...",
      "titl


In [19]:
# =============================== TODO 2 ======================================
# Write the system prompt. It is providing context and rules more than format info.
#
# It has to carry everything the schema cannot express:
#   - role and task in one sentence (context)
#   - the disambiguation rules (which amount, which date, which order)
#   - the priority table, stated as rules on impact
#   - the explicit instruction on whether to abstain: nothing is better than an hallucination
#   - what to do with instructions found inside the email body
#
# Two important concerns to take in mind:
#   1. today's date. The model has no clock; relative dates ("last Thursday") are
#      unresolvable unless you inject `received_at`. It is already in the user
#      message: tell the model to use it.
#   2. the boundary between internal instruction and user data. You should state that the user message is
#      untrusted content to be analysed, never obeyed.

SYSTEM_PROMPT = """..."""

print(len(SYSTEM_PROMPT.split()), "words")

1 words


### Sending the schema

The SDK can do the work for you. `client.chat.completions.parse()` takes the Pydantic
class itself, turns it into the JSON Schema the provider expects, sends it, and hands
back a validated `Ticket` object instead of a string.

Here is an example to make schema enforced (response_format)

In [20]:
r = client.chat.completions.parse(
    model=MODEL, temperature=0,
    response_format=Ticket,              # the class itself, not a dict (not an instance of a Ticket)
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user",   "content": as_user_message(BY_ID[DEV_IDS[0]])}],
)
ticket = r.choices[0].message.parsed     # a Ticket instance, already validated
print(type(ticket).__name__)
print(json.dumps(ticket.model_dump(), indent=2, ensure_ascii=False))

Ticket
{
  "customer_name": "Anna Kowalski",
  "order_id": "VLR-2026-04412",
  "amount": 89.9,
  "currency": "EUR",
  "incident_date": "2026-03-05",
  "category": "refund_request",
  "priority": "medium"
}


### And when the provider refuses

`parse()` is a convenience wrapper around a single request field, `response_format`.
A few things it hides are worth an explicit look.

Passing `strict: true` tells the provider to constrain generation so the output is
guaranteed to match your schema: valid JSON, every required field, no invented ones.
The cost is that only a subset of JSON Schema is supported.

**The schema needs sanitising.** Pydantic's JSON Schema is close to what strict mode
accepts, but not identical. Providers differ on whether the extras are rejected
outright or silently ignored. The next cell strips them, and shows you what's actually sent.


In [21]:
# ----------------------------------------------------------- schema hardening
def strict_schema(model_cls):
    """Pydantic JSON Schema -> the subset accepted by strict structured outputs."""
    s = json.loads(json.dumps(model_cls.model_json_schema()))  # deep copy

    def walk(n):
        if isinstance(n, dict):
            n.pop("default", None)          # not allowed in strict mode
            n.pop("format", None)           # unsupported keyword subset
            if n.get("type") == "object":
                n["additionalProperties"] = False
                n["required"] = list(n.get("properties", {}))
            for v in list(n.values()):
                walk(v)
        elif isinstance(n, list):
            for v in n:
                walk(v)

    walk(s)
    return s

RESPONSE_FORMAT = {"type": "json_schema",
                   "json_schema": {"name": "ticket", "strict": True,
                                   "schema": strict_schema(Ticket)}}
print(json.dumps(RESPONSE_FORMAT, indent=2)[:700])

{
  "type": "json_schema",
  "json_schema": {
    "name": "ticket",
    "strict": true,
    "schema": {
      "properties": {
        "customer_name": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "description": "...",
          "title": "Customer Name"
        },
        "order_id": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "description": "...",
          "title": "Order Id"
        },
        "amount": {
          "anyOf": [
            {
              "ty


Now the same call on all 35 development emails.

In [27]:

MAX_WORKERS = 16          # tune to your rate limit, not to your CPU
_lock, _done = threading.Lock(), 0

def process(eid):
    global _done
    t0 = time.time()
    row = {"id": eid, "pred": None, "latency": None,
           "in_tokens": 0, "out_tokens": 0, "error": None}
    try:
        r = client.chat.completions.create(
            model=MODEL, temperature=0, response_format=RESPONSE_FORMAT,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user",   "content": as_user_message(BY_ID[eid])}],
        )
        row["in_tokens"]  = r.usage.prompt_tokens
        row["out_tokens"] = r.usage.completion_tokens
        try:                            # valid JSON can still be an invalid Ticket
            row["pred"] = Ticket.model_validate_json(
                r.choices[0].message.content).model_dump()
        except Exception as e:
            row["error"] = f"parse:{type(e).__name__}"
    except Exception as e:              # transport / rate limit / 5xx
        row["error"] = f"api:{type(e).__name__}"
    finally:
        row["latency"] = time.time() - t0
        with _lock:
            _done += 1
            print(f"\r{_done}/{len(DEV_IDS)}", end="")
    return row

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    rows = list(ex.map(process, DEV_IDS))   # map preserves input order



35/35

In [28]:
df_v1    = pd.DataFrame(rows).set_index("id")
preds_v1 = df_v1["pred"].to_dict()

In [29]:
print(f"\nparse rate      {df_v1.pred.notna().sum()}/{len(df_v1)}")
print(f"errors          {df_v1.error.value_counts().to_dict()}")
print(f"latency p50/p95 {df_v1.latency.median():.2f}s / {df_v1.latency.quantile(.95):.2f}s")
print(f"tokens          {df_v1.in_tokens.sum()} in / {df_v1.out_tokens.sum()} out"
      f"  ({df_v1.in_tokens.mean():.0f} in per email)")


parse rate      35/35
errors          {}
latency p50/p95 4.38s / 23.68s
tokens          9823 in / 2580 out  (281 in per email)


### Scoring

Accuracy alone is a bad summary. Three numbers matter more, and they are the three you
would quote to an executive committee:

- **hallucination rate** — the fact was absent, the model produced one anyway. This is
  the failure mode that erodes trust, because the output looks perfectly well-formed.
- **omission rate** — the fact was there, the model returned null. Annoying, cheap,
  visible.
- **escalation recall** — of the tickets that genuinely needed fast handling, how many
  were routed as such. This is an efficiency question

In [30]:
NULLABLE   = ["customer_name", "order_id", "amount", "currency", "incident_date"] # The fields that might be null.
ALL_FIELDS = NULLABLE + ["category", "priority"] # Fields that are always present, and the nullable ones.

def eq(field, a, b):
    """Is the predicted value the same fact as the labelled one?"""
    if a is None or b is None:
        return a is None and b is None # e.g. both are null
    if field == "amount":
        try:
            return abs(float(a) - float(b)) < 0.01 # e.g. 99.0 vs 99
        except (TypeError, ValueError):
            return False    # e.g. a schema that types amount as str: "about 90 euros"
    if field == "customer_name":
        return " ".join(str(a).split()).lower() == " ".join(str(b).split()).lower() # e.g. "John  Doe" vs "john doe"
    return str(a).strip().lower() == str(b).strip().lower() # e.g. "urgent" vs "Urgent"

print(len(ALL_FIELDS), "fields compared;", len(NULLABLE), "of them may be null")

7 fields compared; 5 of them may be null


In [31]:
def score(preds, label):
    ok = [i for i in preds if preds[i] is not None] # Only the emails that were successfully parsed.
    rows = [] 
    for f in ALL_FIELDS:
        gt_null   = [i for i in ok if GT_BY_ID[i][f] is None] # The emails where the ground truth is null for this field.
        gt_filled = [i for i in ok if GT_BY_ID[i][f] is not None] # The emails where the ground truth is filled for this field.
        rows.append({
            "field": f, # The field name.
            "accuracy": sum(eq(f, preds[i].get(f), GT_BY_ID[i][f]) for i in ok) / len(ok), # accuracy: the proportion of emails where the predicted value matches the ground truth.
            "hallucination": (sum(preds[i].get(f) is not None for i in gt_null) # The number of emails where the ground truth is null but the prediction is not null.
                              / len(gt_null)) if gt_null else None, 
            "omission": (sum(preds[i].get(f) is None for i in gt_filled) # The number of emails where the ground truth is filled but the prediction is null.
                         / len(gt_filled)) if gt_filled else None,
        })
    per_field = pd.DataFrame(rows).set_index("field")

    hot     = [i for i in ok if GT_BY_ID[i]["priority"] in ("high", "urgent")] # The emails where the ground truth priority is high or urgent.
    recall  = sum(preds[i].get("priority") in ("high", "urgent") for i in hot) / len(hot) # The proportion of emails where the ground truth is high/urgent and the prediction is also high/urgent.
    perfect = sum(all(eq(f, preds[i].get(f), GT_BY_ID[i][f]) for f in ALL_FIELDS) for i in ok) # The number of emails where all fields match the ground truth.
    print(f"--- {label} ---")
    print(f"parse rate            {len(ok)}/{len(preds)}")
    print(f"fully correct emails  {perfect}/{len(preds)} = {perfect/len(preds):.0%}")
    print(f"escalation recall     {recall:.0%}   (high/urgent routed as high/urgent)")
    display(per_field.style.format({"accuracy": "{:.0%}", "hallucination": "{:.0%}",
                                    "omission": "{:.0%}"}, na_rep="-"))
    return per_field

sc_v1 = score(preds_v1, "v1")

--- v1 ---
parse rate            35/35
fully correct emails  14/35 = 40%
escalation recall     56%   (high/urgent routed as high/urgent)


,accuracy,hallucination,omission
field,,,
customer_name,97%,-,0%
order_id,100%,0%,0%
amount,89%,11%,0%
currency,97%,11%,0%
incident_date,74%,89%,0%
category,74%,-,0%
priority,60%,-,0%


## C. Where it is confidently wrong · 25 min

You now have a number. A number is not a diagnosis. Everything useful is in the
residual.

Run the next cell, then **read the failing emails themselves**, not just the table.

In [32]:
rows = []
for eid, p in sorted(preds_v1.items()):
    g = GT_BY_ID[eid]
    if p is None:
        rows.append({"id": eid, "field": "(no parse)", "predicted": "-",
                     "expected": "-", "trap": g.get("trap", "")})
        continue
    for f in ALL_FIELDS:
        if not eq(f, p.get(f), g[f]):
            rows.append({"id": eid, "field": f, "predicted": p.get(f),
                         "expected": g[f], "trap": g.get("trap", "")})

fail_v1 = pd.DataFrame(rows)
print(f"{len(fail_v1)} field errors over {len(preds_v1)} emails")
display(fail_v1)
print("\nerrors by trap type:")
display(fail_v1.trap.value_counts())

38 field errors over 35 emails


,id,field,predicted,expected,trap
0,E04,incident_date,2026-03-10,None,none
1,E04,category,product_defect,general_inquiry,none
2,E04,priority,medium,low,none
3,E05,amount,0.0,None,missing_amount_plus_safety
4,E05,currency,EUR,None,missing_amount_plus_safety
5,E05,category,billing_error,product_defect,missing_amount_plus_safety
6,E05,priority,high,urgent,missing_amount_plus_safety
7,E07,priority,medium,high,none
8,E08,incident_date,2026-03-11,None,none
9,E08,category,product_defect,general_inquiry,none



errors by trap type:


trap
none                          18
missing_amount_plus_safety     4
two_amounts_in_body            4
numeric_distractor             3
amounts_in_words               3
information_starved            2
two_people                     2
quoted_chain_distractor        1
two_order_ids                  1
Name: count, dtype: int64

In [33]:
# Read the emails you got wrong. Do not skip this — the table hides the reason.
for eid in fail_v1.id.unique()[:4]:
    print("=" * 78)
    print(f"{eid}  |  trap = {GT_BY_ID[eid].get('trap')}")
    print(BY_ID[eid]["subject"])
    print(BY_ID[eid]["body"][:700])
    print("--- predicted:", json.dumps(preds_v1[eid], ensure_ascii=False))
    print("--- expected :", json.dumps({k: GT_BY_ID[eid][k] for k in ALL_FIELDS}, ensure_ascii=False))
    print("--- why      :", GT_BY_ID[eid].get("label_note") or "(straightforward case)")

E04  |  trap = none
Question about battery range
Hello,

Before I buy: what is the realistic range of the Velora Trekking 2 in cold weather?
The website says 90 km but I ride in Finland and it is often -10 degrees.

Thanks,
Juhani Virtanen
--- predicted: {"customer_name": "Juhani Virtanen", "order_id": null, "amount": null, "currency": null, "incident_date": "2026-03-10", "category": "product_defect", "priority": "medium"}
--- expected : {"customer_name": "Juhani Virtanen", "order_id": null, "amount": null, "currency": null, "incident_date": null, "category": "general_inquiry", "priority": "low"}
--- why      : (straightforward case)
E05  |  trap = missing_amount_plus_safety
Brake lever snapped
Hello,

The left brake lever on my Velora Urban Pro snapped while riding on 8 March.
Order VLR-2026-03901. The bike is four months old and has done maybe 400 km.
This could have caused an accident on a descent.

I need this handled urgently.

Thomas Berger
--- predicted: {"customer_name": "Thoma

### Now build v2

Three changes, in this order. Do not do them all at once — measure after each, or you
will not know which one worked. That habit is the entire content of Session 5.

**1. Give the model somewhere to put its uncertainty.** Add `requires_human_review:
bool` to the schema, with the trigger list from the spec in its description. A model
that is wrong 8% of the time and flags 100% of its own errors is deployable. A model
that is wrong 4% of the time silently is not. This is the single most important design
move in the notebook.

**2. Force grounding.** Add `amount_evidence: Optional[str]` — the exact substring of
the email that justifies `amount`, or null. Requiring a quote makes fabrication
structurally harder: the model has to produce a span that is either in the text or
visibly not. Watch what it does to the hallucination column.

**3. Patch the specific rules you saw fail.** Tone versus impact, which of two dates,
words-as-numbers. Add them to the system prompt as rules, not as pleading.

> **Field order is not cosmetic.** The model generates left to right, so a field placed
> before `amount` conditions it, and a field placed after is conditioned by it. Putting
> `amount_evidence` *before* `amount` makes it a working step; putting it after makes it
> a post-hoc justification of a number already committed to. Try both if you have time.

In [35]:
# =============================== TODO 3 ======================================
# Schema v2 and prompt v2. This cell only DECLARES them; the next cell runs them.
# Change one thing at a time and re-score.

class TicketV2(BaseModel):
    ...

SYSTEM_PROMPT_V2 = SYSTEM_PROMPT + """

..."""

In [ ]:
# The same loop as v1, with the v2 schema and the v2 prompt.
RESPONSE_FORMAT_V2 = {"type": "json_schema",
                      "json_schema": {"name": "ticket", "strict": True,
                                      "schema": strict_schema(TicketV2)}}
preds_v2 = {}
for n, eid in enumerate(DEV_IDS, 1):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, response_format=RESPONSE_FORMAT_V2,
        messages=[{"role": "system", "content": SYSTEM_PROMPT_V2},
                  {"role": "user",   "content": as_user_message(BY_ID[eid])}],
    )
    try:
        preds_v2[eid] = TicketV2.model_validate_json(r.choices[0].message.content).model_dump()
    except Exception:
        preds_v2[eid] = None
    print(f"\r{n}/{len(DEV_IDS)}", end="")

print()
sc_v2 = score(preds_v2, "v2")

35/35
--- v2 ---
parse rate            35/35
fully correct emails  0/35 = 0%
escalation recall     0%   (high/urgent routed as high/urgent)


,accuracy,hallucination,omission
field,,,
customer_name,0%,-,100%
order_id,20%,0%,100%
amount,26%,0%,100%
currency,26%,0%,100%
incident_date,26%,0%,100%
category,0%,-,100%
priority,0%,-,100%


In [37]:
# v1 vs v2, side by side. This table goes in your report.
cmp = sc_v1[["accuracy", "hallucination"]].join(
    sc_v2[["accuracy", "hallucination"]], lsuffix="_v1", rsuffix="_v2")
cmp["delta_accuracy"] = cmp.accuracy_v2 - cmp.accuracy_v1
display(cmp.style.format("{:.0%}", na_rep="-"))

# Does the review flag actually catch the errors it is supposed to catch?
ok = [i for i, p in preds_v2.items() if p is not None]
wrong   = [i for i in ok if any(not eq(f, preds_v2[i].get(f), GT_BY_ID[i][f]) for f in ALL_FIELDS)]
flagged = [i for i in ok if preds_v2[i].get("requires_human_review")]
caught  = len(set(wrong) & set(flagged))
print(f"\nemails with >=1 error : {len(wrong)}")
print(f"flagged for review    : {len(flagged)}  ({len(flagged)/len(ok):.0%} of volume sent to a human)")
print(f"errors caught by flag : {caught}/{len(wrong)} = {caught/max(len(wrong),1):.0%}")
print(f"errors shipped silently: {len(wrong)-caught}")

,accuracy_v1,hallucination_v1,accuracy_v2,hallucination_v2,delta_accuracy
field,,,,,
customer_name,97%,-,0%,-,-97%
order_id,100%,0%,20%,0%,-80%
amount,89%,11%,26%,0%,-63%
currency,97%,11%,26%,0%,-71%
incident_date,74%,89%,26%,0%,-49%
category,74%,-,0%,-,-74%
priority,60%,-,0%,-,-60%



emails with >=1 error : 35
flagged for review    : 0  (0% of volume sent to a human)
errors caught by flag : 0/35 = 0%
errors shipped silently: 35


That last block is the shape of every real deployment decision. You are not choosing
between "accurate" and "inaccurate". You are choosing a point on a curve: what share of
volume goes to a human, and what share of errors escapes. Move the flag's threshold and
you slide along it.

**For your report:** at your current v2 numbers, what fraction of the 2 000 weekly
emails still needs a human, and what does the system actually save?

## D. Letting the model act · 20 min

Extraction reads. Triage needs to *do*: check the order actually exists, compare the
amount the customer claims with the amount Velora charged, open the ticket.

The mechanism has one property worth internalising:

> **The model never executes anything.** It emits a structured request — a name and
> arguments — and stops. Your code decides whether to honour it, runs it, and hands the
> result back. Function calling is a *protocol*, not an execution engine.

Which means the security boundary is exactly where you put it, and the arguments are
attacker-influenced input. Hold that thought until Part E.

In [38]:
# ---------------------------------------------------------------- your backend
CALL_LOG = []

def lookup_order(order_id: str):
    CALL_LOG.append(("lookup_order", order_id))
    return ORDERS.get(order_id, {"error": "order_not_found", "order_id": order_id})

TICKETS = {}
def create_ticket(order_id, category, priority, amount, summary):
    tid = f"TCK-{len(TICKETS)+1:04d}"
    TICKETS[tid] = dict(order_id=order_id, category=category, priority=priority,
                        amount=amount, summary=summary)
    CALL_LOG.append(("create_ticket", tid))
    return {"ticket_id": tid, "status": "created"}

DISPATCH = {"lookup_order": lookup_order, "create_ticket": create_ticket}
print(len(ORDERS), "orders in the database")

41 orders in the database


In [40]:
# =============================== TODO 4 ======================================
# Describe your two functions to the model. The description fields ARE the prompt:
# they are the only thing the model has to decide when to call what.

TOOLS = [
    {"type": "function", "function": {
        "name": "lookup_order",
        "description": "...",          # <- when should the model reach for this?
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string", "description": "..."}},
                       "required": ["order_id"]},
    }},
    # <- describe create_ticket, with all five arguments and their enums
]

TOOL_SYSTEM_PROMPT = """..."""

In [41]:
# ------------------------------------------------------- the loop (pre-written)
def triage_with_tools(email, max_turns=6, verbose=True):
    messages = [{"role": "system", "content": TOOL_SYSTEM_PROMPT},
                {"role": "user",   "content": as_user_message(email)}]
    for turn in range(max_turns):
        r = client.chat.completions.create(
            model=MODEL, temperature=0, messages=messages,
            tools=TOOLS, tool_choice="auto")
        m = r.choices[0].message

        entry = {"role": "assistant", "content": m.content or ""}
        if m.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in m.tool_calls]
        messages.append(entry)

        if not m.tool_calls:
            if verbose: print(f"[turn {turn}] final: {m.content}")
            return m.content, messages

        for tc in m.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = DISPATCH[tc.function.name](**args)   # <- YOUR code runs, not the model's
            if verbose:
                print(f"[turn {turn}] {tc.function.name}({json.dumps(args, ensure_ascii=False)[:110]})")
                print(f"           -> {json.dumps(result, ensure_ascii=False)[:110]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return None, messages

# E03 is a double-charge claim; the order database disagrees with the customer.
_, trace = triage_with_tools(BY_ID["E03"])
print("\ntickets:", json.dumps(TICKETS, indent=2, ensure_ascii=False))

[turn 0] lookup_order({"order_id": "VLR-2026-04455"})
           -> {"order_id": "VLR-2026-04455", "customer_name": "Sofia Marchetti", "total_charged": 259.0, "currency": "EUR", 
[turn 1] final: I can see your order VLR-2026-04455 was successfully delivered with a total charge of 259.00 EUR. However, you mentioned being charged 249.00 EUR twice, which suggests there may be a discrepancy between the order amount and what was actually charged to your card.

To help you resolve this duplicate charge issue, I recommend:

1. **Contact customer support directly** - They can investigate the payment processing system to see if there was indeed a duplicate transaction
2. **Provide them with both transaction IDs** from your bank statement showing the two identical charges
3. **Keep your bank statement handy** as evidence of the duplicate charges

The customer service team will be able to verify the transactions and process the refund for the duplicate charge. You may want to reach out to them vi

In [42]:
# Run it on a handful and watch the decisions, not just the outputs.
CALL_LOG.clear(); TICKETS.clear()
for eid in ["E04", "E14", "E42"]:
    print("\n" + "=" * 78 + f"\n{eid}: {BY_ID[eid]['subject']}")
    triage_with_tools(BY_ID[eid])

print("\ncall log:", CALL_LOG)


E04: Question about battery range
[turn 0] final: I understand you have a question about the realistic range of the Velora Trekking 2 electric bike in cold weather conditions, specifically around -10°C in Finland. Unfortunately, I don't have access to the technical specifications or performance data for this particular e-bike model.

For accurate information about how cold weather affects the battery range, I'd recommend:

1. Contacting Velora customer support directly - they would have the most detailed technical specifications and real-world performance data
2. Checking e-bike forums or reviews from users in similar cold climates
3. Consulting with a local e-bike dealer who has experience with this model in cold conditions

Generally speaking, lithium-ion batteries (like those used in e-bikes) do experience reduced performance in cold temperatures, so the 90 km range stated on the website is likely based on ideal conditions and may be lower in -10°C weather.

Is there anything else 

**Questions for the report.**

1. On `E04` (a pre-sales question with no order), did the model call `lookup_order`
   anyway? What in your description caused or prevented that?
2. `E14` and `E03` claim amounts that disagree with the order database. Did the model
   notice? What did it write in `summary`?
3. `create_ticket` mutates your system. What stops the model from calling it twenty
   times? Right now: nothing but a prompt. Name two controls you would put in the
   dispatcher — not in the prompt — before this touches production.

## E. The email that attacks you · 10 min

One of the 50 emails is not just a customer email. **Find it before running the next
cell** — the cell only checks your answer, it does not search for you.

Where to look: your v2 predictions. An email whose extraction you cannot explain from
its facts alone, or one your model flagged for review without an obvious reason.
Then read the body.

In [ ]:
INJECTED_ID = ""   # <- the id of the email you found, e.g. "E07"

assert INJECTED_ID, "Find it first. Start from the emails your v2 flagged for review."
print(BY_ID[INJECTED_ID]["body"])
print("\n--- v2 extraction:")
print(json.dumps(preds_v2.get(INJECTED_ID), indent=2, ensure_ascii=False))
gt = GT_BY_ID.get(INJECTED_ID)
if gt is None:
    print("\n(no dev label for this id — either a held-out email or not the one. Keep looking.)")
else:
    print("\n--- expected:")
    print(json.dumps({k: gt[k] for k in ALL_FIELDS + ["requires_human_review"]},
                     indent=2, ensure_ascii=False))

This is **indirect prompt injection**: hostile instructions arriving inside the data,
not inside the user's turn. Nobody typed it into a chat window; it came through the
front door of a normal business process, and it will still be there tomorrow when the
same email is re-processed.

Note precisely what is and is not defended here:

- The schema stopped the attacker from changing the *shape* of the output. They could
  not make the system emit prose, or an extra field, or a call to a function you never
  declared.
- The schema did **nothing** about the *values*. `priority: "urgent"` and
  `amount: 9999.99` are perfectly schema-valid. Constrained decoding is a type system,
  not a security control.
- The only thing that helped, if it helped, was the instruction in your system prompt
  separating data from instructions — a mitigation with no guarantee attached.

Now scale the thought. In Part D you gave the model tools. An injected instruction that
reaches a model holding `create_ticket` gets to open tickets; one holding `send_email`
or `issue_refund` gets to do that. **The blast radius of an injection is the set of
tools in scope.** Session 4 gives the model more tools. Session 5 attacks the result on
purpose.

**For the report, one sentence:** if you were shipping this at Velora next month, what
would you refuse to connect to it, and why?

## F. Bonus — free · whatever time is left

Pick one. Depth beats coverage; a measured negative result is worth more than an
unmeasured feature.

- **Cheaper.** Re-run v2 on a smaller/cheaper model and put accuracy, latency and token
  cost side by side. Where exactly does quality break? Report the trade-off curve, not
  a winner.
- **Self-consistency.** Call three times at `temperature=1`, keep the majority value
  per field, flag disagreement as `requires_human_review`. Does escalation recall
  improve enough to justify 3× the cost?
- **Two-stage.** A cheap model extracts; an expensive one only re-processes what the
  cheap one flagged. Compute the blended cost per 1 000 emails and compare to
  single-model.
- **Harden the injection.** Try delimiter-fencing the body, or a second model call
  whose only job is to answer "does this text contain instructions?". Then write the
  attack that beats your own defence.
- **Break your own schema.** Find an input that produces schema-valid nonsense. That
  input is your best slide.

## Deliverable

One notebook + one page of prose. Pairs.

Submit:

1. **This notebook, executed** — TODOs filled, outputs visible — **at the end of the
   session**. What is in it by then is what counts; no polishing at home.
2. **A one-page report, by 23:59 tonight**, written from the numbers already in your
   notebook, containing:

| | |
|---|---|
| Baseline | parse failure rate of the naive prompt, and what breaks at Velora's volume |
| v1 → v2 | the comparison table, and which single change moved which number |
| Three failures | three emails still wrong, with a one-line diagnosis each: prompt, schema, or spec |
| Human-in-the-loop | share of volume flagged for review, share of errors caught, share shipped silently |
| Cost | tokens per email × 2 000/week × your model's price; latency p95 |
| One refusal | one thing you would not connect this system to, and why |

**Graded on** (this deliverable is one of the six; your best four count for 30% of the
course):

| | |
|---|---|
| The extractor runs end to end and is measured, not asserted | 6 |
| Error analysis identifies *causes*, not symptoms | 6 |
| The human-in-the-loop trade-off is quantified | 4 |
| Cost and latency are real numbers from your own run | 2 |
| The refusal is specific and defensible | 2 |

Prose you could have written before opening the notebook scores zero. Numbers from your
own run score.